# Imports

In [1]:
import numpy as np
import math
from scipy.spatial.transform import Rotation as rotate

from functions.representation import Object, RectangularPrism, find_center_point_LWLC, load_objects
from functions.vectors import cosine_similarity, find_axis_of_rotation_geon_only, find_axis_of_rotation_geon_and_spatcon, same_object
from functions.plotting import add_frame, prepare_rotation_graphs

import plotly.graph_objects as go
import plotly.offline as pyo
import copy

# Initialize Plotly for offline mode in Jupyter Notebook
pyo.init_notebook_mode(connected=True)

# Model Variables

In [ ]:
# MAYBE: main variables are axis difficulty, participant ability/experience w/ MR, participant gender

total_run_time = 0

# timing
production_time = 50                        # 50ms
propositional_difficulty_time = 3           # 1-3 ms --> more difficult on combination rotations
object_encoding_time = 225                  # 150-300ms for each object

# rotation
speed_constant = 3 # idek like 2-4 seems realistic? maybe reduce on repeat --> slower start on combination rotations

# decision
repeat_threshold = 2         # 0-4 repeated steps

# similarity thresholds
geon_alignment_threshold = 0.8  # maybe 0.7 - 0.9
landmark_angle_threshold = 0.99
landmark_distance_threshold = 0.4               # this has to be higher as speed constant gets higher to ensure correct decision (if not there are more errors i thinkie)
object_angle_threshold = 0.975
object_distance_threshold = landmark_distance_threshold * 2

# Model Functions

In [3]:
def calculate_step_size_simple(angular_disparity):
    step_size = speed_constant * math.sqrt(angular_disparity)

    return step_size

# 1. Representation

This is the case I'll model:

![image](test.jpg)

*150 degree diff in pic

In [4]:
r = rotate.from_rotvec([0, 0, np.deg2rad(150)])                               # 150 deg rotation around z-axis
# r = rotate.from_rotvec([np.deg2rad(150), np.deg2rad(60), np.deg2rad(150)])       # crazy rotation :o

# x: right is positive, y: further away is positive, z: up is positive

# create original geons
g1 = RectangularPrism(2, np.array([-1, -1, 0]))             # 45 deg angle front left, no z info
g2 = RectangularPrism(3, np.array([0, 0, -1]))              # down
g3 = RectangularPrism(2, np.array([1, 1, 0]))               # 45 deg angle back right, no z info
g4 = RectangularPrism(1, np.array([1, -1, 0]))              # 45 deg angle front right, no z info

# create original object and relations
original_object = Object(
    geons = [g1,g2,g3,g4],
    landmark_geon_index = 0                    # i.e. landmark is g1
)

# create target geons (same as original, but with rotation applied)
g1 = RectangularPrism(2, r.apply(np.array([-1, -1, 0])))
g2 = RectangularPrism(3, r.apply(np.array([0, 0, -1])))
g3 = RectangularPrism(2, r.apply(np.array([1, 1, 0])))
g4 = RectangularPrism(1, r.apply(np.array([1, -1, 0])))

# # MIRRORED target object
# g1 = RectangularPrism(2, r.apply(np.array([-1, -1, 0])))
# g2 = RectangularPrism(3, r.apply(np.array([0, 0, -1])))
# g3 = RectangularPrism(2, r.apply(np.array([1, 1, 0])))
# g4 = RectangularPrism(1, r.apply(np.array([-1, 1, 0])))

target_object = Object(
    geons = [g1,g2,g3,g4],
    landmark_geon_index = 0                    # i.e. landmark is g1
)

# make copy of original object
original_object_reset = copy.deepcopy(original_object)

# calculate axis of rotation, direction of rotation, and total angular disparity
center_point = np.array([0,0,0])
axis_of_rotation, direction, curr_step_angular_disparity = find_axis_of_rotation_geon_only(original_object, target_object, center_coords=center_point)
_, _, total_angular_disparity = find_axis_of_rotation_geon_and_spatcon(original_object, target_object, center_coords=center_point)

# add time needed for representation
total_run_time = total_run_time + (2 * object_encoding_time)

# 2. Landmarking

In [5]:
# add time needed for landmarking
total_run_time = total_run_time + production_time                                                                 # check geon
total_run_time = total_run_time + production_time + (total_angular_disparity * propositional_difficulty_time)     # check spatial connection
print(total_angular_disparity)
print(total_run_time)

150.28224293627088
1150.8467288088127


# 3. Rotation & 4. Decision (Loop)

In [6]:
repeat_count = -1
same = False
while not same and repeat_count < repeat_threshold:

    # count repeated rotation step
    if not same:
        repeat_count += 1

        if repeat_count > 0:
            print("REPEAT #" + str(repeat_count))
            print(total_run_time)

    # prepare graphs
    axis_animation_fig, overlap_animation_fig, sidebyside_animation_fig = prepare_rotation_graphs(original_object, target_object, axis_of_rotation, center_point, production_time)
    axis_animation = []
    overlap_animation = []
    sidebyside_animation = []

    # set landmark vectors, angular disparity
    original_landmark_geon_vector = original_object.get_landmark_geon().get_vector()
    target_landmark_geon_vector = target_object.get_landmark_geon().get_vector()
    original_spatcon_direction = original_object.get_landmark_spatial_connection().get_vector()
    target_spatcon_direction = target_object.get_landmark_spatial_connection().get_vector()
    total_angular_disparity_part2 = None

    # STEP ONE: GEON ALIGNMENT

    loop_count = 0
    while cosine_similarity(original_landmark_geon_vector, target_landmark_geon_vector) < geon_alignment_threshold:     # checking cosine similarity between target and goal geons

        # find best axis/direction of rotation, angular disparity
        axis_of_rotation, direction, curr_step_angular_disparity = find_axis_of_rotation_geon_only(original_object, target_object, center_coords=center_point, prev_axis=axis_of_rotation, prev_direction=direction, prev_angle=curr_step_angular_disparity, total_angular_disparity=total_angular_disparity)

        # calculate step size based on angular disparity
        # step_size = calculate_step_size_min_jerk_trajectory(curr_step_angular_disparity, total_angular_disparity)
        # step_size = fitts_step(curr_step_angular_disparity, loop_count)
        step_size = calculate_step_size_simple(curr_step_angular_disparity)
        print("CURR_ANGULAR_DISPARITY: " + str(curr_step_angular_disparity) + ", STEP_SIZE: " + str(step_size))

        # apply rotation to original object
        r = rotate.from_rotvec(direction * np.deg2rad(step_size) * axis_of_rotation)        # quaternion representing step size rotation around calculated axis
        original_object.rotate(r)

        # update original geon and spatial connection vectors
        original_landmark_geon_vector = original_object.get_landmark_geon().get_vector()
        original_spatcon_direction = original_object.get_landmark_spatial_connection().get_vector()

        # add animation frames to graphs
        add_frame(axis_animation, original_object, object_name="Original Object", landmark_name="Original Landmarks", axis_of_rotation=axis_of_rotation, object_colour='blue', landmark_colour='purple', axis_colour='green', axis_scale=2)
        original_centerpoint_vec = find_center_point_LWLC(original_object)
        copy_og_obj = copy.deepcopy(original_object)
        copy_og_obj.update_start_coords(copy_og_obj.start_coords - original_centerpoint_vec)
        add_frame(overlap_animation, copy_og_obj, object_name="Original Object", object_colour='blue', axis_scale=2)
        add_frame(sidebyside_animation, copy_og_obj, object_name="Original Object", object_colour='blue', axis_scale=2)

        loop_count += 1
        total_run_time += production_time * 3

        # emergency break
        if loop_count > 100:
            break

    # STEP TWO: GEON AND SPAT CON ALIGNMENT:

    loop_count = 0
    while cosine_similarity(original_landmark_geon_vector, target_landmark_geon_vector) < landmark_angle_threshold or cosine_similarity(original_spatcon_direction, target_spatcon_direction) < landmark_angle_threshold:     # checking cosine similarity between target and goal geons and spatial connections

        # find best axis/direction of rotation, angular disparity
        axis_of_rotation, direction, curr_step_angular_disparity = find_axis_of_rotation_geon_and_spatcon(original_object, target_object, center_coords=center_point, prev_axis=axis_of_rotation, prev_direction=direction, prev_angle=curr_step_angular_disparity, total_angular_disparity=total_angular_disparity)

        # calculate step size based on angular disparity
        if total_angular_disparity_part2 == None:
            total_angular_disparity_part2 = curr_step_angular_disparity
        # step_size = calculate_step_size_min_jerk_trajectory(curr_step_angular_disparity, total_angular_disparity_part2)
        # step_size = fitts_step(curr_step_angular_disparity, loop_count)
        step_size = calculate_step_size_simple(curr_step_angular_disparity)
        print("CURR_ANGULAR_DISPARITY: " + str(curr_step_angular_disparity) + ", STEP_SIZE: " + str(step_size))

        # apply rotation to original object
        r = rotate.from_rotvec(direction * np.deg2rad(step_size) * axis_of_rotation)        # quaternion representing step size rotation around calculated axis
        original_object.rotate(r)

        # update original geon and spatial connection vectors
        original_landmark_geon_vector = original_object.get_landmark_geon().get_vector()
        original_spatcon_direction = original_object.get_landmark_spatial_connection().get_vector()

        # add animation frames to graphs
        add_frame(axis_animation, original_object, object_name="Original Object", landmark_name="Original Landmarks", axis_of_rotation=axis_of_rotation, object_colour='blue', landmark_colour='purple', axis_colour='green', axis_scale=2)
        original_centerpoint_vec = find_center_point_LWLC(original_object)
        copy_og_obj = copy.deepcopy(original_object)
        copy_og_obj.update_start_coords(copy_og_obj.start_coords - original_centerpoint_vec)
        add_frame(overlap_animation, copy_og_obj, object_name="Original Object", object_colour='blue', axis_scale=2)
        add_frame(sidebyside_animation, copy_og_obj, object_name="Original Object", object_colour='blue', axis_scale=2)

        loop_count += 1
        total_run_time += production_time * 3

        # emergency break
        if loop_count > 100:
            break

    axis_animation_fig.frames = axis_animation
    axis_animation_fig.show()
    overlap_animation_fig.frames = overlap_animation
    overlap_animation_fig.show()
    sidebyside_animation_fig.frames = sidebyside_animation
    sidebyside_animation_fig.show()

    # check overall similarity

    same, similarity_check_run_time = same_object(original_object, target_object, object_angle_threshold, total_angular_disparity, production_time, propositional_difficulty_time)
    total_run_time += similarity_check_run_time

    if not same:

        # reset original object
        original_object = original_object_reset
        axis_of_rotation, direction, angle = find_axis_of_rotation_geon_only(original_object, target_object, center_coords=center_point)
        total_angular_disparity = angle

print("DECISION:")
if same:
    print("same")
else:
    print("different")

print("\nRUN TIME:\n" + str(round(total_run_time/1000, 3)) + " seconds")

CURR_ANGULAR_DISPARITY: 130.89339464913093, STEP_SIZE: 34.322595354112984
CURR_ANGULAR_DISPARITY: 113.42726330541306, STEP_SIZE: 31.95067088104282
CURR_ANGULAR_DISPARITY: 83.76400094894849, STEP_SIZE: 27.45680259135314
CURR_ANGULAR_DISPARITY: 60.95741979159914, STEP_SIZE: 23.4225698445835
CURR_ANGULAR_DISPARITY: 32.847361328907546, STEP_SIZE: 17.193785271433626
CURR_ANGULAR_DISPARITY: 15.653576057473929, STEP_SIZE: 11.869380123547536


DECISION:
same

RUN TIME:
4.705 seconds
